# Latent Gradient Steering v2

This notebook implements latent reasoning steering via gradient ascent in the continuous hidden states of a language model. It leverages contrastive exemplars to structurally map empirical scientific arguments in the model's internal geometry.

In [ ]:
# 0. Install and Import Required Libraries
!pip install -U bitsandbytes>=0.46.1 transformers accelerate

import random
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running experiment on: {device}")

Running experiment on: cuda


In [ ]:
# 1. Load Larger Model with 4-Bit Quantization
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_name)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)
base_model.eval()
print("Loaded Model")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded Model


In [ ]:
# 2. Contrastive Exemplars & Prompt
positive_exemplars = [
    "Nuclear detonations created a globally synchronous radiocarbon spike in tree rings, leaving a permanent marker.",
    "The K-T extinction event is proved by global iridium layers in geological strata, an element rare in Earth's crust.",
    "Ancient human agriculture caused detectable methane anomalies in Antarctic ice cores long before the industrial revolution.",
    "Roman lead mining operations left global isotopic heavy metal pollution preserved in Arctic ice layers."
]

negative_exemplars = [
    "If a secret society detonated nuclear weapons centuries ago, they used energy shielding leaving no trace.",
    "Dinosaurs were wiped out by an asteroid that vaporized leaving zero rocks, dust, or chemicals behind.",
    "Early humans farmed rice in water, and the Silurian period also had a lot of water.",
    "A dinosaur civilization's metal cities simply rusted away into dust over millions of years."
]

prompt = (
    "In response to the Silurian Hypothesis, evaluate whether an industrial non-human "
    "civilization millions of years ago would leave physical evidence in the geological record. "
    "Provide a counterargument based on environmental proxies."
)

print("Exemplars and prompt loaded.")

Exemplars and prompt loaded.


In [ ]:
# 3. Extract Centroids in Base LLM Hidden Space
def get_centroid(texts):
    states = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt").to(device)
            out = base_model(**inputs, output_hidden_states=True)

            # Mean-pool across all tokens in the sequence to capture semantic meaning
            hidden_states = out.hidden_states[-1].squeeze(0) # [seq_len, hidden_dim]
            mean_pooled = torch.mean(hidden_states, dim=0)   # [hidden_dim]
            states.append(mean_pooled.unsqueeze(0))

    return torch.mean(torch.cat(states, dim=0), dim=0)

ideal_centroid = get_centroid(positive_exemplars)
corrupt_centroid = get_centroid(negative_exemplars)

norm_layer = getattr(base_model.model, "norm", torch.nn.Identity())
print("Centroids extracted with mean pooling.")

Centroids extracted with mean pooling.


In [ ]:
# 4. Control Generation
print("[DEBUG Control] Control generation function loaded.")
def generate_control(prompt_text, max_tokens=250):
    print("\n[DEBUG Control] Starting control generation...")
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    outputs = base_model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    print("[DEBUG Control] Control generation completed.")
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# 5. Treatment Generation (Latent Gradient Ascent) with Trajectories
print("[DEBUG Control] Treatment generation function loaded.")
def generate_treatment(prompt_text, max_tokens=250, steps=8, lr=0.5, l2_weight=1.0, sim_scale=100.0, track_trajectories=True):
    print(f"\n[DEBUG Treatment] Starting treatment generation (Max tokens: {max_tokens}, Steps/token: {steps})...")
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    generated_ids = inputs.input_ids.clone()

    all_trajectories = []

    print("[DEBUG Treatment] Cloning centroids to float32...")
    # Centroids are in float32 for stable gradient calculations
    ideal_c = ideal_centroid.clone().to(torch.float32)
    corrupt_c = corrupt_centroid.clone().to(torch.float32)

    for token_idx in range(max_tokens):
        print(f"\n[DEBUG Treatment] --- Processing Token {token_idx + 1}/{max_tokens} ---")

        with torch.no_grad():
            outputs = base_model(input_ids=generated_ids, output_hidden_states=True)
            initial_hidden = outputs.hidden_states[-1][:, -1, :]
        print(f"[DEBUG Treatment] Forward pass complete. Initial hidden state shape: {initial_hidden.shape}")

        # Cast latent state to float32 to prevent 16-bit underflow of micro-gradients
        h_latent = initial_hidden.clone().detach().to(torch.float32).requires_grad_(True)
        optimizer = torch.optim.Adam([h_latent], lr=lr)

        token_trajectory = []

        print(f"[DEBUG Treatment] Beginning {steps} gradient ascent steps...")
        for step in range(steps):
            optimizer.zero_grad()

            # Calculate similarities and distances in float32
            sim_good = F.cosine_similarity(h_latent, ideal_c.unsqueeze(0))
            sim_bad = F.cosine_similarity(h_latent, corrupt_c.unsqueeze(0))
            manifold_dist = torch.norm(h_latent - initial_hidden.to(torch.float32), p=2)

            # Apply sim_scale to balance similarity gradients against the L2 norm vector
            objective = ((sim_good - (0.5 * sim_bad)) * sim_scale) - (l2_weight * manifold_dist)
            loss = -objective

            if track_trajectories:
                token_trajectory.append({
                    "step": step + 1,
                    "loss": loss.item(),
                    "sim_good": sim_good.item(),
                    "sim_bad": sim_bad.item(),
                    "manifold_dist": manifold_dist.item()
                })

            loss.backward()
            optimizer.step()

            # Print first, middle, and last step to avoid too much spam, but verify progress
            if step == 0 or step == steps//2 or step == steps - 1:
                print(f"    [Step {step + 1}/{steps}] Loss: {loss.item():.4f} | Sim_Good: {sim_good.item():.4f} | Manifold_Dist: {manifold_dist.item():.4f}")

        print(f"[DEBUG Treatment] Gradient steps finished for Token {token_idx + 1}.")

        if track_trajectories:
            all_trajectories.append(token_trajectory)

        # Cast back to the model's original dtype (float16) before generation
        with torch.no_grad():
            normed_h = norm_layer(h_latent.to(initial_hidden.dtype))
            logits = base_model.lm_head(normed_h)
            next_token_id = torch.argmax(logits, dim=-1, keepdim=True)

            # Decode the selected token to show user the exact word being produced
            new_word = tokenizer.decode(next_token_id[0])
            print(f"[DEBUG Treatment] Generated token ID: {next_token_id.item()} -> Word: '{new_word}'")

        generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

        if next_token_id.item() == tokenizer.eos_token_id:
            print(f"[DEBUG Treatment] EOS token reached at token {token_idx + 1}. Breaking loop.")
            break

    print("\n[DEBUG Treatment] Treatment generation completed successfully.")
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True), all_trajectories

In [ ]:
# 6. Compare Control vs. Treatment and Output Trajectories
print("\n--- STEP 6: COMPARISON AND TRAJECTORY ANALYSIS ---\n")

print("Generating Control Response...")
control_output = generate_control(prompt, max_tokens=150)

print("\nGenerating Treatment Response (with Trajectory Tracking)...")
treatment_output, trajectories = generate_treatment(prompt, max_tokens=150, track_trajectories=True)

print("\n=== CONTROL OUTPUT (Unsteered) ===")
print(control_output)
print("\n" + "="*50 + "\n")

print("=== TREATMENT OUTPUT (Steered) ===")
print(treatment_output)
print("\n" + "="*50 + "\n")

print("=== RESULT ANALYSIS ===")
print("Standard generation (Control) relies on standard autoregressive CoT, which may hallucinate reasoning or "
      "provide logically unfaithful responses based on its baseline priors. \n"
      "In contrast, the Treatment generation bypasses reliance purely on text prediction. By geometrically steering "
      "the model's internal continuous hidden state vectors towards the ideal, evidence-based centroid and away "
      "from the corrupted, anecdotal centroid, the Treatment trajectory anchors the output in empirical rigor. "
      "The result maps a structural path of rigorous reasoning before translating it to discrete tokens.")
print("\n" + "="*50 + "\n")

print("=== RAW TRAJECTORY VALUES (First 3 Tokens) ===")
# Show the gradient trajectory values for the first 3 generated tokens to verify script functionality
for token_idx, token_traj in enumerate(trajectories[:3]):
    print(f"\nToken {token_idx + 1} optimization trajectory:")
    for step_data in token_traj:
        print(f"  Step {step_data['step']}: Loss: {step_data['loss']:.4f} | "
              f"Sim_Good: {step_data['sim_good']:.4f} | "
              f"Sim_Bad: {step_data['sim_bad']:.4f} | "
              f"L2_Manifold_Dist: {step_data['manifold_dist']:.4f}")

if len(trajectories) > 3:
    print("\n... (remaining token trajectories omitted for brevity) ...\n")


--- STEP 6: COMPARISON AND TRAJECTORY ANALYSIS ---

Generating Control Response...

[DEBUG Control] Starting control generation...
[DEBUG Control] Control generation completed.

Generating Treatment Response (with Trajectory Tracking)...
f
[DEBUG Treatment] Starting treatment generation (Max tokens: {max_tokens}, Steps/token: {steps})...
[DEBUG Treatment] Cloning centroids to float32...

[DEBUG Treatment] --- Processing Token 1/150 ---
[DEBUG Treatment] Forward pass complete. Initial hidden state shape: torch.Size([1, 4096])
[DEBUG Treatment] Beginning 8 gradient ascent steps...
    [Step 1/8] Loss: -11.5020 | Sim_Good: 0.1768 | Manifold_Dist: 0.0000
    [Step 5/8] Loss: 7.3398 | Sim_Good: 0.1578 | Manifold_Dist: 17.1822
    [Step 8/8] Loss: -2.4988 | Sim_Good: 0.1906 | Manifold_Dist: 10.1849
[DEBUG Treatment] Gradient steps finished for Token 1.
[DEBUG Treatment] Generated token ID: 781 -> Word: '
'

[DEBUG Treatment] --- Processing Token 2/150 ---
[DEBUG Treatment] Forward pass comp

In [ ]:
# 7. Generate DPO Dataset
def generate_dpo_dataset(prompts, output_file="dpo_dataset.jsonl"):
    dpo_data = []

    for idx, prompt_text in enumerate(prompts):
        print(f"\nProcessing prompt {idx + 1}/{len(prompts)} for DPO dataset...")

        # 1. Generate the Rejected response (Default/Unsteered)
        rejected_text = generate_control(prompt_text, max_tokens=150)

        # 2. Generate the Chosen response (Steered/Optimized)
        chosen_text, _ = generate_treatment(
            prompt_text,
            max_tokens=150,
            track_trajectories=False # Turn off logging to speed up generation
        )

        # 3. Append to DPO structure
        dpo_data.append({
            "prompt": prompt_text,
            "chosen": chosen_text,
            "rejected": rejected_text
        })

    # Export to JSONL
    with open(output_file, 'w') as f:
        for item in dpo_data:
            f.write(json.dumps(item) + '\n')

    print(f"Successfully saved {len(prompts)} preference pairs to {output_file}")

# Example Usage:
# prompts_list = ["Evaluate the Silurian Hypothesis...", "Explain the Fermi Paradox..."]
# generate_dpo_dataset(prompts_list)

# 1. Define a list of prompts you want to run through the pipeline
prompts_list = [
    "In response to the Silurian Hypothesis, evaluate whether an industrial non-human civilization millions of years ago would leave physical evidence in the geological record. Provide a counterargument based on environmental proxies.",
    "Explain the Fermi Paradox and evaluate the hypothesis that advanced civilizations inevitably destroy themselves before achieving interstellar travel.",
    "Analyze the geological evidence for the Younger Dryas impact hypothesis. Contrast the empirical data with astronomical models of comet fragmentation."
]

# 2. Call the function to generate the dataset
print("Starting DPO Dataset Generation...")
generate_dpo_dataset(prompts_list, output_file="dpo_dataset.jsonl")
print("Finished!")

Starting DPO Dataset Generation...

Processing prompt 1/3 for DPO dataset...

[DEBUG Control] Starting control generation...
[DEBUG Control] Control generation completed.
f
[DEBUG Treatment] Starting treatment generation (Max tokens: {max_tokens}, Steps/token: {steps})...
[DEBUG Treatment] Cloning centroids to float32...

[DEBUG Treatment] --- Processing Token 1/150 ---
[DEBUG Treatment] Forward pass complete. Initial hidden state shape: torch.Size([1, 4096])
[DEBUG Treatment] Beginning 8 gradient ascent steps...
    [Step 1/8] Loss: -11.5020 | Sim_Good: 0.1768 | Manifold_Dist: 0.0000
    [Step 5/8] Loss: 7.3398 | Sim_Good: 0.1578 | Manifold_Dist: 17.1822
    [Step 8/8] Loss: -2.4988 | Sim_Good: 0.1906 | Manifold_Dist: 10.1849
[DEBUG Treatment] Gradient steps finished for Token 1.
[DEBUG Treatment] Generated token ID: 781 -> Word: '
'

[DEBUG Treatment] --- Processing Token 2/150 ---
[DEBUG Treatment] Forward pass complete. Initial hidden state shape: torch.Size([1, 4096])
[DEBUG Treat